# 02. Layout Detection (Model A - Step 1)

Surya를 사용해 스캔 페이지에서 영역을 감지하고 분류합니다.

**감지 레이블:** Title / Section-header / Text / Formula / Table / Picture / Caption / List-item / Page-header / Page-footer

In [ ]:
!pip install "surya-ocr==0.12.1" -q

In [ ]:
import fitz
import io
import os
import json
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

PDF_PATH = "../data/sample/2015개정경제수학-광주교육청.pdf"
OUTPUT_DIR = "../outputs/layout"
os.makedirs(OUTPUT_DIR, exist_ok=True)

doc = fitz.open(PDF_PATH)
print(f"총 페이지 수: {len(doc)}")

총 페이지 수: 198


In [2]:
def page_to_image(doc, page_num, dpi=200):
    page = doc[page_num]
    pix = page.get_pixmap(dpi=dpi)
    return Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")

## 1. 모델 로드

첫 실행 시 모델 가중치를 다운로드합니다 (~1GB).

In [ ]:
import torch
from surya.layout import LayoutPredictor

device = "cuda" if torch.cuda.is_available() else "cpu"
predictor = LayoutPredictor(device=device)
print(f"모델 로드 완료 (device={device})")

## 2. 단일 페이지 테스트

In [ ]:
TEST_PAGE = 11  # 0-indexed (12페이지)

img = page_to_image(doc, TEST_PAGE)
results = predictor([img])
layout = results[0]

print(f"[{TEST_PAGE+1}페이지] 감지된 요소: {len(layout.bboxes)}개\n")
for i, bbox in enumerate(layout.bboxes):
    conf = getattr(bbox, 'confidence', None)
    conf_str = f"{conf:.2f}" if conf is not None else "n/a"
    print(f"  [{i+1}] {bbox.label:<16} confidence={conf_str}  bbox={[round(x) for x in bbox.bbox]}")

## 3. 결과 시각화

In [ ]:
LABEL_COLORS = {
    "Title":          "#e74c3c",
    "Section-header": "#e67e22",
    "Text":           "#2ecc71",
    "Formula":        "#9b59b6",
    "Table":          "#3498db",
    "Picture":        "#1abc9c",
    "Caption":        "#f39c12",
    "List-item":      "#27ae60",
    "Page-header":    "#95a5a6",
    "Page-footer":    "#bdc3c7",
}

def visualize_layout(img, layout, title="Layout Detection"):
    fig, ax = plt.subplots(1, 1, figsize=(10, 14))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(title, fontsize=13)

    w, h = img.size
    for bbox in layout.bboxes:
        x1, y1, x2, y2 = bbox.bbox
        color = LABEL_COLORS.get(bbox.label, "#ecf0f1")
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor=color, alpha=0.15
        )
        ax.add_patch(rect)
        ax.text(
            x1, y1 - 4, f"{bbox.label} ({bbox.confidence:.2f})",
            color=color, fontsize=7.5, fontweight="bold",
            bbox=dict(facecolor="white", alpha=0.6, pad=1, edgecolor="none")
        )

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"page_{TEST_PAGE+1}_layout.png"), dpi=150, bbox_inches="tight")
    plt.show()

visualize_layout(img, layout, title=f"{TEST_PAGE+1}페이지 Layout Detection")

## 4. 구조화 JSON 변환

bbox를 0~1 정규화 좌표로 저장합니다. (이미지 크기에 독립적)

In [ ]:
def layout_to_json(page_num, img, layout):
    w, h = img.size
    elements = []
    for i, bbox in enumerate(layout.bboxes):
        x1, y1, x2, y2 = bbox.bbox
        elements.append({
            "id": f"p{page_num+1}_e{i+1}",
            "type": bbox.label.lower().replace("-", "_"),
            "bbox_px": [round(x1), round(y1), round(x2), round(y2)],
            "bbox_norm": [round(x1/w, 4), round(y1/h, 4), round(x2/w, 4), round(y2/h, 4)],
            "confidence": round(bbox.confidence, 4),
            "text": None,       # 03_text_ocr.ipynb 에서 채움
            "latex": None,      # 04_formula_ocr.ipynb 에서 채움
            "description": None # Model B 에서 채움
        })

    # 읽기 순서: 위→아래, 같은 y면 왼→오른쪽
    elements.sort(key=lambda e: (e["bbox_px"][1], e["bbox_px"][0]))
    for i, e in enumerate(elements):
        e["reading_order"] = i + 1

    return {
        "page_id": page_num + 1,
        "image_size": {"width": w, "height": h},
        "element_count": len(elements),
        "elements": elements
    }


page_json = layout_to_json(TEST_PAGE, img, layout)

out_path = os.path.join(OUTPUT_DIR, f"page_{TEST_PAGE+1}_layout.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(page_json, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {out_path}")
print(json.dumps(page_json, ensure_ascii=False, indent=2)[:800])

## 5. 여러 페이지 배치 처리

In [ ]:
TEST_PAGES = range(9, 14)  # 10~14페이지 (01_ocr_parser와 동일 범위)

all_results = []

for page_num in TEST_PAGES:
    print(f"[{page_num+1}페이지] 처리 중...", end=" ")
    img = page_to_image(doc, page_num)
    layout = predictor([img])[0]
    page_json = layout_to_json(page_num, img, layout)
    all_results.append(page_json)

    label_summary = {}
    for e in page_json["elements"]:
        label_summary[e["type"]] = label_summary.get(e["type"], 0) + 1
    print(f"{page_json['element_count']}개 감지 → {label_summary}")

batch_out = os.path.join(OUTPUT_DIR, "batch_layout.json")
with open(batch_out, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f"\n배치 저장 완료: {batch_out}")